<a href="https://colab.research.google.com/github/Md-iftikher/Deep_learning_practice_10/blob/main/practice_pytorch_manual_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.__version__)
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

In [ ]:
df.shape

In [ ]:
# to drop a colum
df.drop(columns=['Unnamed: 32','id'],inplace=True)

In [ ]:
df.head()

train test split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:, 1:],df.iloc[:, 0], test_size=0.2, random_state=42)

**Scaling**

In [ ]:
scaler = StandardScaler()
X_train= scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train

In [ ]:
y_train

# Label Encoder




In [ ]:
encoder= LabelEncoder()
y_train= encoder.fit_transform(y_train)
y_test= encoder.transform(y_test)
y_train

# Numpy Arrays to pytorch tensors

In [ ]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [ ]:
X_train_tensor.shape

# Defining the model

In [ ]:
from math import log
class MySimpleNN():

  def __init__(self,X):

    self.weight = torch.rand((X.shape[1],1), dtype= torch.float64, requires_grad=True)
    self.bias = torch.zeros(1, dtype= torch.float64, requires_grad=True)#

  def forward(self,X):
    z= torch.matmul(X,self.weight)+self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  # loss function binary cross entrophy as this is binary classifiaction

  def loss_function(self,y_pred,y):
    # clamp prediction to avoid log(0)
    epsilon= 1e-7
    y_pred= torch.clamp(y_pred, epsilon , 1-epsilon)

    # calculate loss
    loss = -(y_train_tensor*torch.log(y_pred)+ (1-y_train_tensor)*torch.log(1-y_pred)).mean()
    return loss



# Trainign Pipline

In [ ]:
# Important parameter
learning_rate = 0.1
epochs =25

In [ ]:
# create Model manually
model = MySimpleNN(X_train_tensor)

# define loops
for epoch in range(epochs):

    # forward pass
    y_pred=model.forward(X_train_tensor)

    # loss calcualtion
    loss = model.loss_function(y_pred,y_train_tensor)
    # print(f'Epoch: {epoch}, Loss: {loss.item()}')

    # backward pass for calculating gradient
    loss.backward()

    # update parameter usign funtion no_grad to stop gradient tacking for this
    with torch.no_grad():
      model.weight -= learning_rate * model.weight.grad
      model.bias -= learning_rate * model.bias.grad


    # zero gradient
    model.weight.grad.zero_()
    model.bias.grad.zero_()

    # print loss in each epoch
    print(f'Epoch: {epoch+1}, Loss: {loss.item()}')





In [ ]:
model.bias

In [ ]:
model.weight

# Evaluation

In [ ]:
#model evaluation
with torch.no_grad():
  y_pred=model.forward(X_test_tensor)
  y_pred= (y_pred> 0.9)
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy}')